# 1D chaotic Kuramoto–Sivashinsky: interactive space–time audit

This notebook uses the canonical one-dimensional periodic form

$$u_t+u u_x+u_{xx}+u_{xxxx}=0,\qquad x\in[0,60).$$

This follows the homogeneous chaotic-system setting of [Pathak et al.](https://arxiv.org/abs/1710.07313): $L=60$ and saved cadence $\Delta t=0.25$. A burn-in removes the hand-chosen initial transient before recording.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

case_name = "kuramoto_sivashinsky"
smoke_mode = os.environ.get("PHYCOFLOW_VIZ_SMOKE", "0") == "1"
seed = 7
n = 64 if smoke_mode else 256
domain_length = 60.0
solver_dt = 0.01 if smoke_mode else 0.05
burn_in_time = 0.05 if smoke_mode else 50.0
record_time = 0.05 if smoke_mode else 100.0
save_every = 1 if smoke_mode else 5  # normal output cadence: 0.25
rng = np.random.default_rng(seed)


In [2]:
x = np.linspace(0.0, domain_length, n, endpoint=False)
dx = domain_length / n
wave = 2.0 * np.pi * np.fft.fftfreq(n, d=dx)
modes = np.fft.fftfreq(n) * n
dealias = np.abs(modes) <= n / 3
linear = wave**2 - wave**4

def nonlinear_term(u_hat):
    u = np.fft.ifft(u_hat).real
    return -0.5j * wave * np.fft.fft(u**2) * dealias

def initial_condition():
    base = np.cos(2.0 * np.pi * x / domain_length) * (1.0 + np.sin(2.0 * np.pi * x / domain_length))
    noise_hat = np.fft.fft(rng.standard_normal(n)) * np.exp(-0.02 * wave**4) * dealias
    perturbation = np.fft.ifft(noise_hat).real
    perturbation -= perturbation.mean()
    return base + 0.05 * perturbation / perturbation.std()

# Kassam–Trefethen ETDRK4 coefficients.
roots = np.exp(1j * np.pi * (np.arange(1, 17) - 0.5) / 16)
lr = solver_dt * linear[:, None] + roots
E = np.exp(solver_dt * linear)
E2 = np.exp(0.5 * solver_dt * linear)
Q = solver_dt * np.real(np.mean((np.exp(lr / 2.0) - 1.0) / lr, axis=-1))
f1 = solver_dt * np.real(np.mean((-4.0 - lr + np.exp(lr) * (4.0 - 3.0 * lr + lr**2)) / lr**3, axis=-1))
f2 = solver_dt * np.real(np.mean((2.0 + lr + np.exp(lr) * (-2.0 + lr)) / lr**3, axis=-1))
f3 = solver_dt * np.real(np.mean((-4.0 - 3.0 * lr - lr**2 + np.exp(lr) * (4.0 - lr)) / lr**3, axis=-1))

u_hat = np.fft.fft(initial_condition()) * dealias
burn_steps = int(round(burn_in_time / solver_dt))
record_steps = int(round(record_time / solver_dt))
snapshots, times_list = [], []
for step in range(burn_steps + record_steps + 1):
    if step >= burn_steps and (step - burn_steps) % save_every == 0:
        snapshots.append(np.fft.ifft(u_hat).real)
        times_list.append((step - burn_steps) * solver_dt)
    if step == burn_steps + record_steps:
        break
    Nv = nonlinear_term(u_hat)
    a = E2 * u_hat + Q * Nv
    Na = nonlinear_term(a)
    b = E2 * u_hat + Q * Na
    Nb = nonlinear_term(b)
    c = E2 * a + Q * (2.0 * Nb - Nv)
    Nc = nonlinear_term(c)
    u_hat = (E * u_hat + f1 * Nv + 2.0 * f2 * (Na + Nb) + f3 * Nc) * dealias

times = np.asarray(times_list)
fields = np.asarray(snapshots)[:, None, :]  # [time, channel, x]


In [3]:
def spatial_pde_terms(field):
    field_hat = np.fft.fft(field)
    advection = np.fft.ifft(0.5j * wave * np.fft.fft(field**2) * dealias).real
    uxx = np.fft.ifft(-(wave**2) * field_hat).real
    uxxxx = np.fft.ifft(wave**4 * field_hat).real
    return advection + uxx + uxxxx

residuals, time_terms = [], []
for index in range(1, times.size - 1):
    ut = (fields[index + 1, 0] - fields[index - 1, 0]) / (times[index + 1] - times[index - 1])
    residuals.append(ut + spatial_pde_terms(fields[index, 0]))
    time_terms.append(ut)
relative_residual = np.linalg.norm(residuals) / (np.linalg.norm(time_terms) + 1.0e-12)
diagnostics = {
    "relative_pde_residual": float(relative_residual),
    "finite": bool(np.isfinite(fields).all()),
    "initial_std": float(fields[0, 0].std()),
    "final_std": float(fields[-1, 0].std()),
    "temporal_std": float(fields[:, 0].std(axis=0).mean()),
}
diagnostics


{'relative_pde_residual': 0.005482635841046905,
 'finite': True,
 'initial_std': 1.1905213004689992,
 'final_std': 1.113052967137064,
 'temporal_std': 1.1322231038728254}

In [4]:
def draw_frame(frame_index, axes):
    for axis in axes.ravel():
        axis.clear()
    field = fields[frame_index, 0]
    axes[0, 0].imshow(fields[:, 0], origin="lower", aspect="auto", extent=(0, domain_length, times[0], times[-1]), cmap="RdBu_r")
    axes[0, 0].axhline(times[frame_index], color="black", lw=1)
    axes[0, 0].set(xlabel="x", ylabel="t", title="Chaotic space–time field")
    axes[0, 1].plot(x, field, color="tab:blue")
    axes[0, 1].set(xlim=(0, domain_length), title=f"u(x), t={times[frame_index]:.2f}")
    axes[0, 2].hist(field, bins=30, density=True, color="tab:blue", alpha=0.8)
    axes[0, 2].set_title("Value PDF (Wasserstein input)")
    power = np.abs(np.fft.rfft(field - field.mean()))**2 / field.size**2
    axes[1, 0].semilogy(np.arange(power.size)[1:], power[1:] + 1.0e-18, color="tab:orange")
    axes[1, 0].set_title("1D power spectrum")
    threshold = np.median(field)
    axes[1, 1].fill_between(x, 0, field > threshold, step="mid", alpha=0.7)
    axes[1, 1].set(xlim=(0, domain_length), ylim=(0, 1.1), title="Superlevel components")
    ux = np.fft.ifft(1j * wave * np.fft.fft(field)).real
    axes[1, 2].plot(field, ux, lw=0.8)
    axes[1, 2].set(xlabel="u", ylabel="u_x", title="Spatial phase portrait")
    return []

dashboard = None
if not smoke_mode:
    frame_indices = np.unique(np.linspace(0, times.size - 1, min(times.size, 31), dtype=int))
    figure, dashboard_axes = plt.subplots(2, 3, figsize=(12, 7), dpi=72, constrained_layout=True)
    animation = FuncAnimation(figure, lambda i: draw_frame(i, dashboard_axes), frames=frame_indices, interval=180, repeat=True)
    plt.close(figure)
    dashboard = HTML(animation.to_jshtml(default_mode="once"))
    display(dashboard)
